# Phase 0 · Gate 1 — Base-model capability probe

**Question:** can a small open model hold a multi-step oncology reasoning + tool-use chain
together *well enough to be worth training*?

**Why first:** the whole project post-trains this model. If the base is too weak, no amount
of RFT/DPO saves it. This decision shapes the architecture — make it before building anything.

**Run on Kaggle:** New Notebook → Settings → Accelerator = GPU T4 x2, **Internet = On**
(needs phone verification) → Run all. ~5–10 min. See phases/PHASE0_preflight_gates.md.

> Self-contained: needs no project data, no API key. Tools are simulated — I test whether
> the model *calls* them and follows the output format, not real grounding (that is Phase 3).


## 1 · Confirm the GPU attached


In [ ]:
!nvidia-smi


## 2 · Install
Only transformers + accelerate. **Do not reinstall torch** — Kaggle ships it matched to the
GPU driver. If you get an import/version error after this, Run → Restart & Clear Cell Outputs,
then re-run from here. (For a 4-bit model you also need `bitsandbytes` — see cell 3.)


In [ ]:
!pip -q install -U 'transformers>=4.44' 'accelerate>=0.34'


## 3 · Load the policy model  (modular — swap with one line)
Change **`MODEL_KEY`** to swap models; fp16 vs 4-bit is handled automatically and the model is
pinned to a single T4 (avoids multi-GPU split overhead).

**Escalation ladder:** start `1.5b`; if tool use is still inconsistent *with the tightened
prompt below*, set `3b`; use `7b` (4-bit) only if 3b is still weak.

| key | model | fits T4 | notes |
|---|---|---|---|
| `1.5b` | Qwen2.5-1.5B-Instruct | fp16, trivial | fastest |
| `3b`   | Qwen2.5-3B-Instruct   | fp16, comfortable | ~1.5–2x slower |
| `7b`   | Qwen2.5-7B-Instruct   | 4-bit only | needs `!pip install bitsandbytes` |


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # classic resumable download; HF 'Xet' backend stalls on flaky nets

import torch, time, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda', torch.cuda.is_available())

# ---- model registry: swap models by changing MODEL_KEY only ----------------
import glob
MODELS = {
    '1.5b': dict(id='Qwen/Qwen2.5-1.5B-Instruct', load_in_4bit=False),
    '3b':   dict(id='Qwen/Qwen2.5-3B-Instruct',   load_in_4bit=False),
    '7b':   dict(id='Qwen/Qwen2.5-7B-Instruct',   load_in_4bit=True),  # 4-bit to fit T4
}
MODEL_KEY = '3b'            # <-- change THIS ONE LINE to swap the model
# For instant load, attach the matching size via model_sources in kernel-metadata.json.
# ----------------------------------------------------------------------------

def resolve_local(key):
    # find an attached Kaggle Model mount (qwen2.5 <key>-instruct); path casing/version vary
    for cj in glob.glob('/kaggle/input/**/config.json', recursive=True):
        p = os.path.dirname(cj)
        pl = p.lower()
        if 'qwen2.5' in pl and f'{key}-instruct' in pl:
            return p
    return None

def load_model(key):
    cfg = MODELS[key]
    if os.path.isdir('/kaggle/input'):
        print('inputs mounted:', os.listdir('/kaggle/input'))
    local = resolve_local(key)
    src = local or cfg['id']
    print('source:', 'LOCAL mount' if local else 'HF download', '=', src)
    tok = AutoTokenizer.from_pretrained(src)
    kw = dict(device_map={'': 0})              # single T4; avoids split overhead
    if cfg['load_in_4bit']:
        try:
            import bitsandbytes  # noqa: F401
        except ImportError:
            raise RuntimeError('Run  !pip install bitsandbytes  once, then rerun this cell.')
        kw['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16)
    else:
        kw['dtype'] = torch.float16            # T4 = fp16 (Turing has no native bf16)
    model = AutoModelForCausalLM.from_pretrained(src, **kw)
    print('loaded', key, '| 4bit', cfg['load_in_4bit'], '->', model.device)
    return tok, model

tok, model = load_model(MODEL_KEY)


## 4 · System prompt + three probe cases
Assistive decision-support framing (not an authoritative 'expert' persona — that hurts
calibration and suppresses deferral). Tools are mandated before claims; the output must end
in a fixed Recommendation / Confidence / Defer block.


In [ ]:
SYSTEM = """You are a decision-support assistant for a molecular tumor board. You assist
clinicians and do not replace them. You defer when the evidence is insufficient.

TOOLS (write each call on its own line, exactly this format):
  TOOL: variant_lookup(gene, variant)
  TOOL: guideline_lookup(gene, tumor_type)
  TOOL: trial_search(gene, tumor_type)

RULES:
1. You MUST call the relevant tools BEFORE any factual claim about a variant, guideline,
   or trial. Do not answer from memory.
2. Tie every claim to a tool result and name the tool it came from. If a needed fact is
   not available from a tool, say so rather than inventing it.
3. Reason in explicit, numbered steps.
4. Do not overstate certainty. A lower confidence with a deferral beats a confident guess.
5. End with EXACTLY this block and nothing after it:
   Recommendation: <ranked therapy options, or "defer">
   Confidence: <0.00-1.00>
   Defer: <yes/no - one-line reason>
"""

CASES = [
    'Lung adenocarcinoma, stage IV, treatment-naive. Alteration: EGFR p.L858R.',
    'Lung adenocarcinoma, stage IV. Alteration: EML4-ALK fusion. One prior line of chemo.',
    'Lung adenocarcinoma. Alteration: KRAS p.G12C. Prior platinum doublet.',
]


In [ ]:
def run_case(text, max_new_tokens=800):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': text}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    gen = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return gen, dt

for i, c in enumerate(CASES, 1):
    gen, dt = run_case(c)
    print(f'\n{"="*80}\nCASE {i}  ({dt:.1f}s)\n{c}\n{"-"*80}\n{gen}')


## 5 · Gate decision (you score this by hand)

The gate is about **structure**, not medical correctness (a small model will not be a great
oncologist — that is what retrieval grounding + post-training fix). Score each case 0/1/2:

| Criterion | Look for |
|---|---|
| Reasoning structure | distinct, ordered numbered steps |
| **Tool discipline** | `TOOL:` calls with sensible args **before** any factual claim |
| Output format | ends in the Recommendation / Confidence / Defer block |
| Deferral sense | defers instead of inventing when evidence is missing |

**Decision:**
- **All three comply (tools + format) → CONTINUE** with the current `MODEL_KEY`.
- **Still 1–2 of 3 ignoring tools → ESCALATE**: set `MODEL_KEY='3b'` in cell 3, rerun cell 3
  and cell 4b, and re-score. (Then `7b` only if 3b is still weak.)

Record the decision + one sentence of reasoning — it is a pre-registered architectural choice.
